# 標記樣本拍攝（在 JetBot 上跑）

拍校正用的照片。**兩種照片要分開拍，用途不一樣：**

| 檔名前綴 | 距離 | 用途 |
|---|---|---|
| `close` | 約 0.3 m 特寫 | **量顏色**。貼紙要夠大才框得準 |
| `0.5m` `0.8m` `1.5m` `2.0m` | 站立全身 | **驗證偵測距離**與像素大小 |

---

### 拍之前先確認三件事

1. **攝影機角度已經固定**。角度一改，所有數字都要重來
2. **解析度是 300×300**，與 `live_demo_following_sm.ipynb` 一致。
   標記偵測跑在原始相機畫面上，不是避障模型那條 224×168 的鏈
3. **存 PNG，不要存 JPEG**。JPEG 的色度抽樣（4:2:0）會把色彩解析度砍半，
   對只有幾個像素的色塊是災難性的。下面的程式已經寫死 PNG

拍完把整個 `samples/` 資料夾複製回筆電，跑 `analyze_marker.py`。

In [ ]:
import os

import cv2
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

SAVE_DIR = 'samples'
if not os.path.isdir(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# ⚠️ 這兩個數字必須與 live_demo_following_sm.ipynb 的 Camera.instance() 完全一致。
#
# 校正量的是「標記在畫面上佔幾個像素」,解析度一改,所有門檻都作廢。
#
# 為什麼是 300x300 而不是避障用的 224x168:
#   marker.detect() 跑在 execute() 拿到的**原始相機畫面**上,而 object_following
#   的相機設成 300x300(官方值,因為 SSD MobileNet 吃 300x300 輸入)。
#   避障模型的 224x168 是 preprocess() 裡另外 resize 出來的,與標記偵測無關。
#
# 另外:相機擷取是 816x616(4:3),縮放成 300x300 會水平壓扁,
# 所以垂直與水平的「每像素代表幾公分」不一樣(約 1.9cm vs 2.5cm)。
# 這不影響偵測,但看 --check 的像素數時心裡要有底。
CAM_W, CAM_H = 300, 300

camera = Camera.instance(width=CAM_W, height=CAM_H)
print('相機解析度:%dx%d —— 確認與 live_demo_following_sm.ipynb 相同' % (CAM_W, CAM_H))

# 顯示放大兩倍純粹是為了肉眼看得清楚,存檔存的仍是原始解析度
image_widget = widgets.Image(format='jpeg', width=CAM_W * 2, height=CAM_H * 2)
camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'),
                              transform=bgr8_to_jpeg)
display(image_widget)

### 拍攝清單

每個情境改一次「檔名前綴」再按拍照，檔名會自動編號。

**量顏色（前綴 `close`）—— 5~8 張就夠**
- 一般室內燈光下 2 張
- 靠窗、光線比較亮 2 張
- 陰影裡、光線比較暗 2 張
- 貼紙稍微傾斜（模擬走路時腿的角度）2 張

光線變化一定要涵蓋，不然門檻會只在拍攝當下那個亮度成立。

**驗證距離（前綴 `0.5m` / `0.8m` / `1.5m` / `2.0m`）—— 每個距離 3~4 張**
- 目標者站定、正面 1 張
- 側身 1 張
- 走路中途（兩腿交錯的那一瞬間）1~2 張 ← **最重要，這是最容易失敗的狀況**

**如果霧面和亮面兩種材質都買了**，各拍一輪，前綴加上 `matte` / `gloss`
（例如 `gloss_close`、`matte_1.5m`），分開跑 `--pick` 就能直接比過曝比例。

In [ ]:
prefix_widget = widgets.Text(value='close', description='檔名前綴')
count_widget = widgets.IntText(value=0, description='資料夾張數', disabled=True)
snap_button = widgets.Button(description='拍一張', button_style='success')
undo_button = widgets.Button(description='刪掉上一張', button_style='danger')

_last = {'path': None}


def _refresh_count():
    count_widget.value = len([f for f in os.listdir(SAVE_DIR)
                              if f.lower().endswith('.png')])


def save_one(_):
    prefix = prefix_widget.value.strip() or 'sample'
    n = 1
    while True:
        path = os.path.join(SAVE_DIR, '%s_%02d.png' % (prefix, n))
        if not os.path.exists(path):
            break
        n += 1
    # camera.value 就是 BGR8 的 numpy 陣列，imwrite 直接吃，不要先轉色彩空間
    cv2.imwrite(path, camera.value)
    _last['path'] = path
    _refresh_count()
    print('存了 %s' % path)


def undo_one(_):
    path = _last['path']
    if path and os.path.exists(path):
        os.remove(path)
        _last['path'] = None
        _refresh_count()
        print('刪掉 %s' % path)
    else:
        print('沒有可刪的（只能刪剛拍的那一張）')


snap_button.on_click(save_one)
undo_button.on_click(undo_one)
_refresh_count()

display(widgets.VBox([
    prefix_widget,
    widgets.HBox([snap_button, undo_button]),
    count_widget,
]))

### 收尾

拍完一定要跑下面這格把相機釋放掉，否則其他 notebook（包括 `live_demo_following_sm.ipynb`）
會開不起來，錯誤訊息通常看不出是這個原因。

萬一忘了跑就關掉 notebook，到終端機執行 Kernel → Restart，或重開 JetBot 的 Jupyter。

In [ ]:
camera_link.unlink()
camera.stop()
print('相機已釋放。要繼續拍請從第一格重跑。')